In [16]:
import sys, os
from pathlib import Path

def add_repo_path():
    """
    stock_forecast 프로젝트 루트를 자동 탐색하고,
    해당 경로를 sys.path에 추가하여 import 오류를 방지합니다.
    """
    # __file__이 정의된 경우 (일반 .py 파일)
    if '__file__' in globals():
        current = Path(__file__).resolve().parent
    else:
        # Jupyter Notebook이나 대화형 환경
        current = Path.cwd()

    # 현재 디렉토리와 상위 디렉토리들을 탐색
    for parent in [current] + list(current.parents):
        # DATA 폴더가 존재하는 경로를 찾으면 sys.path에 추가
        if (parent / "DATA").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            print(f"[INFO] Project root added to sys.path: {parent}")
            return str(parent)

    # 만약 못 찾을 경우 대비 - fallback 경로 지정
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        print(f"[WARNING] Using fallback path: {fallback}")
        return fallback

    raise FileNotFoundError("❌ DATA 폴더를 찾을 수 없습니다.")

# 경로 추가 실행
project_root = add_repo_path()

# korea_fs_loader.py

# -------------------------------------------------------------------
# 0. 환경설정
# -------------------------------------------------------------------

import os
import io
import time
import zipfile
import datetime as dt
from typing import List, Dict, Optional

import requests
import pandas as pd
import pymysql
import xml.etree.ElementTree as ET
from DATA.stock_invest_function import get_db_host

# ============================================================
# 0. 사용자 설정
# ============================================================

API_KEY = "여기에_DART_API_KEY_입력"

# DB_INFO = {
#     "host": "192.168.0.230",
#     "port": 3307,
#     "user": "investar",
#     "password": "PASSWORD",
#     "database": "investar",
# }

TARGET_TABLE = "korea_fs_data_from_DART"


# ============================================================
# 1. 상장사 목록 로드
# ============================================================

def load_corp_code(api_key: str, cache_path="dart_corp_codes.csv") -> pd.DataFrame:
    if os.path.exists(cache_path):
        return pd.read_csv(cache_path, dtype=str)

    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    resp = requests.get(url, params={"crtfc_key": api_key})
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        xml_name = [x for x in z.namelist() if x.endswith(".xml")][0]
        with z.open(xml_name) as f:
            tree = ET.parse(f)

    rows = []
    for elem in tree.getroot().findall("list"):
        rows.append({
            "corp_code": elem.findtext("corp_code"),
            "corp_name": elem.findtext("corp_name"),
            "stock_code": elem.findtext("stock_code"),
        })

    df = pd.DataFrame(rows, dtype=str)
    df = df[df["stock_code"].notna() & (df["stock_code"] != "")]
    df.to_csv(cache_path, index=False, encoding="utf-8-sig")
    return df


def get_corp_info(corp_df: pd.DataFrame, ticker: str) -> Optional[Dict[str, str]]:
    ticker = str(ticker).zfill(6)
    row = corp_df.loc[corp_df["stock_code"] == ticker]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        "corp_code": r["corp_code"],
        "corp_name": r["corp_name"],
        "stock_code": ticker,
    }


# ============================================================
# 2. 분기 재무제표 수집
# ============================================================

def get_dart_fs_quarterly(api_key: str, corp_code: str, start_year: int, end_year: int,
                          fs_div="CFS") -> pd.DataFrame:
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

    reprt_codes = {
        "11013": "Q1",
        "11012": "Q2",
        "11014": "Q3",
        "11011": "Q4",
    }

    all_rows = []

    for year in range(start_year, end_year + 1):
        for rc in reprt_codes.keys():
            params = {
                "crtfc_key": api_key,
                "corp_code": corp_code,
                "bsns_year": str(year),
                "reprt_code": rc,
                "fs_div": fs_div,
                "page_no": 1,
                "page_count": 1000,
            }
            r = requests.get(url, params=params)
            data = r.json()

            if data.get("status") != "000":
                continue

            rows = data.get("list", [])
            for row in rows:
                row["reprt_code"] = rc
                row["quarter"] = reprt_codes[rc]
                all_rows.append(row)

            time.sleep(0.15)

    if not all_rows:
        return pd.DataFrame()

    raw = pd.DataFrame(all_rows)

    # 날짜 생성
    def make_date(row):
        y = int(row["bsns_year"])
        rc = row["reprt_code"]
        if rc == "11013": return pd.Timestamp(y, 3, 31)
        if rc == "11012": return pd.Timestamp(y, 6, 30)
        if rc == "11014": return pd.Timestamp(y, 9, 30)
        if rc == "11011": return pd.Timestamp(y, 12, 31)
        return pd.NaT

    raw["date"] = raw.apply(make_date, axis=1)
    return raw


# ============================================================
# 3. 확장된 계정 alias
# ============================================================

ACCOUNT_ALIASES = {
    # 손익
    "sales": ["매출", "수익(매출)", "매출액", "Revenue", "영업수익"],
    "cogs": ["매출원가"],
    "gross_profit": ["매출총이익", "Gross profit"],
    "op_income": ["영업이익", "Operating profit"],
    "net_income": ["당기순이익", "순이익", "Net income"],

    # EPS
    "diluted_eps": ["희석주당순이익", "Diluted earnings"],

    # 자산
    "total_assets": ["자산총계", "총자산"],
    "current_assets": ["유동자산"],
    "cash": ["현금및현금성자산"],
    "st_financial": ["단기금융상품"],
    "receivables": ["매출채권"],
    "inventories": ["재고자산"],
    "noncurrent_assets": ["비유동자산"],
    "ppe": ["유형자산"],
    "intangibles": ["무형자산"],

    # 부채/자본
    "total_liab": ["부채총계", "총부채"],
    "current_liab": ["유동부채"],
    "trade_payables": ["매입채무"],
    "st_borrowings": ["단기차입금"],
    "noncurrent_liab": ["비유동부채"],
    "equity": ["자본총계", "지배기업 소유주지분"],

    # 현금흐름
    "op_cf": ["영업활동현금흐름"],
    "cce_increase": ["현금및현금성자산의 증가", "현금및현금성자산의증가(감소)"],
}


# ============================================================
# 4. 일반 계정 추출 함수
# ============================================================

def _get_amount(fs_df: pd.DataFrame, key: str) -> Optional[float]:
    aliases = ACCOUNT_ALIASES.get(key, [])
    if not aliases:
        return None

    mask = False
    for a in aliases:
        m = fs_df["account_nm"].astype(str).str.contains(a, na=False, regex=False)
        if "account_id" in fs_df.columns:
            m = m | fs_df["account_id"].astype(str).str.contains(a, na=False, regex=False)
        mask = mask | m   # ✅ 누적 OR

    sub = fs_df[mask]
    if sub.empty:
        return None

    vals = pd.to_numeric(
        sub["thstrm_amount"].astype(str).str.replace(",", ""),
        errors="coerce"
    ).dropna()
    if vals.empty:
        return None

    return float(vals.sum())



# ============================================================
# 5. 배당금 지급 전용 함수
# ============================================================

def _get_dividend_paid(fs_df: pd.DataFrame) -> Optional[float]:
    """
    배당금 '지급'만 정확하게 찾아서 합산해 반환
    """

    # 1) 현금흐름표(CF)에서 DividendsPaid 계정 우선
    if "sj_div" in fs_df.columns:
        mask_cf = fs_df["sj_div"].astype(str).str.contains("CF", na=False)
    else:
        mask_cf = True

    mask_div = (
        fs_df["account_id"].astype(str).str.contains("DividendsPaid", na=False)
        | fs_df["account_nm"].astype(str).str.contains("배당금 지급", na=False)
    )

    sub = fs_df[mask_cf & mask_div]

    # 2) 못 찾으면 "배당" 들어간 계정 전체 fallback
    if sub.empty:
        sub = fs_df[fs_df["account_nm"].astype(str).str.contains("배당", na=False)]

    if sub.empty:
        return None

    vals = pd.to_numeric(sub["thstrm_amount"].astype(str).str.replace(",", ""), errors="coerce").dropna()
    if vals.empty:
        return None

    total = vals.sum()
    return float(abs(total))


# ============================================================
# 6. 분기별 재무지표 계산 (long-format)
# ============================================================

def compute_quarterly_indicators(fs_raw: pd.DataFrame) -> pd.DataFrame:
    if fs_raw.empty:
        return pd.DataFrame()

    fs_raw["thstrm_amount"] = pd.to_numeric(
        fs_raw["thstrm_amount"].astype(str).str.replace(",", ""), errors="coerce"
    )

    records = []

    group_cols = ["corp_code", "corp_name", "date"]

    for (corp_code, corp_name, date), grp in fs_raw.groupby(group_cols):

        # 원천 계정
        sales = _get_amount(grp, "sales")
        cogs = _get_amount(grp, "cogs")
        gross_profit = _get_amount(grp, "gross_profit")
        op_income = _get_amount(grp, "op_income")
        net_income = _get_amount(grp, "net_income")
        diluted_eps = _get_amount(grp, "diluted_eps")

        total_assets = _get_amount(grp, "total_assets")
        cur_assets = _get_amount(grp, "current_assets")
        cash = _get_amount(grp, "cash")
        st_fin = _get_amount(grp, "st_financial")
        recv = _get_amount(grp, "receivables")
        inv = _get_amount(grp, "inventories")
        nca = _get_amount(grp, "noncurrent_assets")
        ppe = _get_amount(grp, "ppe")
        intan = _get_amount(grp, "intangibles")

        total_liab = _get_amount(grp, "total_liab")
        cur_liab = _get_amount(grp, "current_liab")
        payables = _get_amount(grp, "trade_payables")
        st_borr = _get_amount(grp, "st_borrowings")
        noncur_liab = _get_amount(grp, "noncurrent_liab")
        equity = _get_amount(grp, "equity")

        op_cf = _get_amount(grp, "op_cf")
        cce_inc = _get_amount(grp, "cce_increase")

        # 배당 전용 함수
        dividend_paid = _get_dividend_paid(grp)

        def safe(a, b):
            if a is None or b in (None, 0):
                return None
            return float(a) / float(b)

        indicators = {
            # 원자료
            "Sales": sales,
            "COGS": cogs,
            "Gross_Profit": gross_profit,
            "Operating_Income": op_income,
            "Net_Income": net_income,
            "Diluted_EPS": diluted_eps,

            "Total_Assets": total_assets,
            "Current_Assets": cur_assets,
            "Cash": cash,
            "ST_Financial": st_fin,
            "Receivables": recv,
            "Inventories": inv,
            "Noncurrent_Assets": nca,
            "PPE": ppe,
            "Intangibles": intan,

            "Total_Liabilities": total_liab,
            "Current_Liabilities": cur_liab,
            "Trade_Payables": payables,
            "ST_Borrowings": st_borr,
            "Noncurrent_Liabilities": noncur_liab,
            "Total_Equity": equity,

            "Operating_CF": op_cf,
            "CashEquivalents_Change": cce_inc,
            "Dividend_Paid": dividend_paid,

            # 재무비율
            "GPM": safe(gross_profit, sales),
            "OPM": safe(op_income, sales),
            "NIM": safe(net_income, sales),
            "ROA": safe(net_income, total_assets),
            "ROE": safe(net_income, equity),
            "Debt_Ratio": safe(total_liab, equity),
            "Current_Ratio": safe(cur_assets, cur_liab),
            "OCF_to_Sales": safe(op_cf, sales),
            "OCF_to_Assets": safe(op_cf, total_assets),
            "Payout_Ratio": safe(dividend_paid, net_income),
        }

        for k, v in indicators.items():
            if v is None:
                continue
            records.append({
                "date": date.date(),
                "company_name": corp_name,
                "ticker": corp_code,
                "indicator": k,
                "value": float(v),
            })

    return pd.DataFrame(records)


# ============================================================
# 7. DB 업로드
# ============================================================

def upload_indicators_to_db(df: pd.DataFrame, db_info: Dict, table_name=TARGET_TABLE):
    if df.empty:
        print("업로드할 데이터 없음.")
        return

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=False,
    )

    sql = f"""
        INSERT INTO {table_name} (date, company_name, ticker, indicator, value)
        VALUES (%s,%s,%s,%s,%s)
        ON DUPLICATE KEY UPDATE
            company_name=VALUES(company_name),
            value=VALUES(value)
    """

    try:
        with conn.cursor() as cur:
            rows = list(df.itertuples(index=False, name=None))
            cur.executemany(sql, rows)
        conn.commit()
        print(f"업로드 완료: {len(df)} rows")
    finally:
        conn.close()

[INFO] Project root added to sys.path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [17]:

API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"
corp_df = load_corp_code(API_KEY)

# # 2) 삼성전자 정보 찾기
# info = get_corp_info(corp_df, "000660")
# if info is None:
#     raise ValueError("삼성전자(005930) corp_code를 찾을 수 없습니다.")
#
# corp_code = info["corp_code"]
# corp_name = info["corp_name"]
# print(f"[INFO] corp_code={corp_code}, corp_name={corp_name}")
#
# # 3) 분기 재무제표 수집
# fs_raw = get_dart_fs_quarterly(
#     api_key=API_KEY,
#     corp_code=corp_code,
#     start_year=2015,
#     end_year=2025,
#     fs_div="CFS",
# )
# if fs_raw.empty:
#     print("[WARN] 재무제표 데이터가 없습니다.")
#     exit()
#
# # compute_quarterly_indicators 가 필요로 하는 열 추가
# fs_raw["corp_code"] = corp_code
# fs_raw["corp_name"] = corp_name
#
# print("[INFO] raw rows:", len(fs_raw))
#
# # 4) 분기별 재무비율 long-format 생성
# df_ind = compute_quarterly_indicators(fs_raw)
# print(df_ind.head())
#
# # upload_to_db(df_samsung, db_info)

[INFO] corp_code=00164779, corp_name=SK하이닉스
[INFO] raw rows: 6669
         date company_name    ticker         indicator         value
0  2015-12-31       SK하이닉스  00164779             Sales  4.022444e+13
1  2015-12-31       SK하이닉스  00164779              COGS  1.051535e+13
2  2015-12-31       SK하이닉스  00164779      Gross_Profit  8.282645e+12
3  2015-12-31       SK하이닉스  00164779  Operating_Income  5.336100e+12
4  2015-12-31       SK하이닉스  00164779        Net_Income  2.256226e+13


In [18]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}


# upload_indicators_to_db(df_ind, db_info)

업로드 완료: 1202 rows


In [19]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import random
import gc
import traceback
import datetime as dt
from tqdm import tqdm
from typing import Dict

import FinanceDataReader as fdr  # ✅ FDR에서 현재 상장 종목 코드 가져오기


# ----------------------------------------------------------------------
# 1. HTTP 세션 (재시도 + User-Agent 설정)
# ----------------------------------------------------------------------
def create_robust_session():
    """안정적인 HTTP 세션 생성 (재시도 + User-Agent)"""
    session = requests.Session()

    # 재시도 전략
    retry_strategy = Retry(
        total=3,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=10,
        pool_maxsize=20,
    )

    session.mount("http://", adapter)
    session.mount("https://", adapter)

    # DART가 봇으로 오해하지 않도록 명시적인 User-Agent 사용
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (DART-FS-Collector/1.0; +stox1224@email.com)"
    })

    return session


# ----------------------------------------------------------------------
# 2. DART 분기 재무제표 수집 함수 (분기별 실패만 스킵, 티커 전체는 유지)
# ----------------------------------------------------------------------
def get_dart_fs_quarterly(api_key: str, corp_code: str, start_year: int, end_year: int,
                          fs_div="CFS", verbose=False, session=None) -> pd.DataFrame:
    """
    DART API로부터 분기별 재무제표 데이터 수집

    Args:
        api_key: DART API 키
        corp_code: 기업 고유번호
        start_year: 시작 연도
        end_year: 종료 연도
        fs_div: 재무제표 구분 (CFS: 연결, OFS: 개별)
        verbose: 상세 로그 출력 여부
        session: requests.Session 객체 (재사용)

    Returns:
        재무제표 데이터프레임
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

    reprt_codes = {
        "11013": "Q1",
        "11012": "Q2",
        "11014": "Q3",
        "11011": "Q4",
    }

    all_rows = []
    api_error_count = 0

    # 세션이 없으면 기본 requests 사용
    if session is None:
        session = requests

    for year in range(start_year, end_year + 1):
        for rc in reprt_codes.keys():
            params = {
                "crtfc_key": api_key,
                "corp_code": corp_code,
                "bsns_year": str(year),
                "reprt_code": rc,
                "fs_div": fs_div,
                "page_no": 1,
                "page_count": 1000,
            }

            # 재시도 로직
            max_retries = 5
            data = None

            for attempt in range(max_retries):
                try:
                    r = session.get(url, params=params, timeout=30)
                    data = r.json()
                    break  # 성공하면 루프 탈출

                except (ConnectionResetError,
                        requests.exceptions.ConnectionError,
                        requests.exceptions.HTTPError,
                        requests.exceptions.Timeout,
                        requests.exceptions.RequestException) as e:

                    if attempt < max_retries - 1:
                        # 지수 백오프: 5초, 10초, 20초, 40초
                        wait_time = 5 * (2 ** attempt)
                        if verbose:
                            print(f"      [재시도 {attempt+1}/{max_retries}] "
                                  f"{year}-{reprt_codes[rc]}: {wait_time}초 대기... ({e})")
                        time.sleep(wait_time)
                    else:
                        if verbose:
                            print(f"      [실패] {year}-{reprt_codes[rc]}: 최대 재시도 초과 - 이 분기 스킵")
                        data = None  # ❗ 이 분기만 건너뜀 (함수 전체 return 안 함)

                except Exception as e:
                    if verbose:
                        print(f"      [예외] {year}-{reprt_codes[rc]}: {str(e)[:100]}")
                    data = None
                    break

            # data가 없으면 이 (year, rc)만 스킵
            if data is None:
                continue

            status = data.get("status")

            # 디버깅: API 응답 상태 확인
            if verbose and status != "000":
                print(f"    [API] {year}-{reprt_codes[rc]}: status={status}, msg={data.get('message', 'N/A')}")

            # 013: 데이터 없음 (정상)
            # 000: 정상
            if status not in ["000", "013"]:
                api_error_count += 1
                if verbose:
                    print(f"    [WARNING] Unexpected status: {status}")

            if status != "000":
                # 013 같은 데이터 없음은 그냥 스킵
                continue

            rows = data.get("list", [])
            for row in rows:
                row["reprt_code"] = rc
                row["quarter"] = reprt_codes[rc]
                all_rows.append(row)

            # 대기 시간: 1.0~2.0초 랜덤
            time.sleep(random.uniform(1.0, 2.0))

    if verbose and api_error_count > 0:
        print(f"    [API 오류 횟수: {api_error_count}]")

    if not all_rows:
        return pd.DataFrame()

    raw = pd.DataFrame(all_rows)

    # 날짜 생성
    def make_date(row):
        y = int(row["bsns_year"])
        rc = row["reprt_code"]
        if rc == "11013": return pd.Timestamp(y, 3, 31)
        if rc == "11012": return pd.Timestamp(y, 6, 30)
        if rc == "11014": return pd.Timestamp(y, 9, 30)
        if rc == "11011": return pd.Timestamp(y, 12, 31)
        return pd.NaT

    raw["date"] = raw.apply(make_date, axis=1)
    return raw


# ----------------------------------------------------------------------
# 3. 전체 상장사 수집 + DB 업로드
#    - FDR KRX 기준 현재 상장 + ETF/ETN/REIT/SPAC 제거
#    - DB에 저장되는 ticker = FDR의 6자리 코드
# ----------------------------------------------------------------------
def collect_and_upload_all_companies(
    api_key: str,
    db_info: Dict,
    start_year: int = 2013,
    end_year: int = 2025,
    fs_div: str = "CFS",
    batch_size: int = 15,
    table_name: str = "korea_fs_data_from_DART",
    listed_only: bool = True,        # DART 기준 상장 플래그
    verbose_first_n: int = 3,
    try_ofs_fallback: bool = True,
    batch_rest_time: int = 60,
    use_fdr_filter: bool = True      # ✅ FDR 기반 현재 상장 일반주식만 사용
):
    """
    모든 상장사의 재무데이터를 수집하고 배치 단위로 DB에 업로드
    """

    # 안정적인 세션 생성
    session = create_robust_session()
    print(f"[INFO] HTTP 세션 생성 완료 (재시도 로직 + User-Agent 설정)")

    # 1) DART 상장사 목록 로드
    print("=" * 70)
    print("[STEP 1] DART 기업 목록 로드 중...")

    # load_corp_code 함수는 사용자가 이미 정의한 것으로 가정
    corp_df = load_corp_code(api_key)

    # (옵션) DART 데이터 내에서 상장사만 1차 필터링
    if listed_only:
        corp_df = corp_df[
            corp_df["stock_code"].notna() &
            (corp_df["stock_code"] != "") &
            (corp_df["stock_code"].str.strip() != "")
        ].copy()
        print(f"[INFO] DART 기준 상장사만 1차 필터링 완료")
        print(f"      → 남은 기업 수: {len(corp_df)}")

    # 2) FinanceDataReader로 현재 상장사만 필터링 + ETF/ETN/REIT/SPAC 제거
    if use_fdr_filter:
        print("\n[STEP 1-2] FinanceDataReader에서 현재 상장 종목 코드 로드 중 (KRX)...")
        try:
            fdr_df = fdr.StockListing("KRX")  # KOSPI + KOSDAQ + KONEX + 기타
            fdr_df["Code"] = fdr_df["Code"].astype(str).str.zfill(6)

            # ETF / ETN / REIT / SPAC 제거
            exclude_types = ["ETF", "ETN", "REIT", "SPAC"]

            if "Type" in fdr_df.columns:
                before_type = len(fdr_df)
                fdr_df = fdr_df[~fdr_df["Type"].isin(exclude_types)].copy()
                after_type = len(fdr_df)
                print(f"[INFO] Type 기반 필터링 (ETF/ETN/REIT/SPAC 제거)")
                print(f"      → 필터 전: {before_type}개, 필터 후: {after_type}개")
            else:
                # 백업 플랜: 종목명 기준으로 ETF/ETN/리츠/스팩 추정 제거
                print("[WARNING] FDR 데이터에 'Type' 컬럼이 없어 Name 기반 필터를 사용합니다.")
                name_pattern = r"ETF|ETN|리츠|리트|스팩|SPAC"
                before_type = len(fdr_df)
                fdr_df = fdr_df[
                    ~fdr_df["Name"].str.contains(name_pattern, case=False, na=False)
                ].copy()
                after_type = len(fdr_df)
                print(f"[INFO] Name 기반 필터링 (ETF/ETN/리츠/스팩 추정 제거)")
                print(f"      → 필터 전: {before_type}개, 필터 후: {after_type}개")

            fdr_codes = set(fdr_df["Code"].tolist())
            print(f"[INFO] FDR KRX 현재 상장 일반 주식 수(필터 후): {len(fdr_codes)}")

            # DART corp_df의 stock_code도 6자리로 통일
            corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

            before_cnt = len(corp_df)
            corp_df = corp_df[corp_df["stock_code"].isin(fdr_codes)].copy()
            after_cnt = len(corp_df)

            print(f"[INFO] FDR 기준 현재 상장 일반 주식으로 2차 필터링 완료")
            print(f"      → 필터 전: {before_cnt}개, 필터 후: {after_cnt}개")
        except Exception as e:
            print(f"[WARNING] FinanceDataReader 기반 필터링 실패: {e}")
            print("          → FDR 필터를 건너뛰고 DART 상장사 목록만 사용합니다.")
    else:
        print("\n[STEP 1-2] use_fdr_filter=False → FinanceDataReader 필터를 사용하지 않습니다.")

    total_companies = len(corp_df)
    print(f"\n[INFO] 최종 대상 기업 수: {total_companies}개 (현재 상장 일반 주식 기준)")

    # 상장사 목록 미리보기
    print("\n[상장사 샘플]")
    print(corp_df[["corp_name", "stock_code"]].head(10))
    print()

    # 2) 통계 변수 초기화
    success_count = 0
    fail_count = 0
    no_data_count = 0
    cfs_to_ofs_count = 0
    failed_companies = []
    processed_count = 0

    # 3) 배치 단위로 처리
    print("=" * 70)
    print(f"[STEP 2] 재무데이터 수집 시작")
    print(f"[설정] 배치크기={batch_size}, 배치간휴식={batch_rest_time}초")
    print(f"[설정] fs_div={fs_div}, OFS대체={try_ofs_fallback}")
    print(f"[설정] API대기시간=1.0~2.0초(랜덤)")
    print("=" * 70)

    total_batches = (total_companies + batch_size - 1) // batch_size

    for batch_idx in range(0, total_companies, batch_size):
        batch_end = min(batch_idx + batch_size, total_companies)
        batch_df = corp_df.iloc[batch_idx:batch_end]
        current_batch_num = batch_idx // batch_size + 1

        print(f"\n{'='*70}")
        print(f"[BATCH {current_batch_num}/{total_batches}] 처리 중: "
              f"{batch_idx+1}~{batch_end}/{total_companies}")
        print(f"{'='*70}")

        # 배치 내 모든 기업의 데이터를 모을 리스트
        batch_indicators = []

        # 배치 내 각 기업 처리
        for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc="기업 처리"):
            ticker = row["stock_code"]      # ✅ FDR Code (6자리, DB에 저장할 ticker)
            corp_name = row["corp_name"]
            corp_code = row["corp_code"]
            processed_count += 1

            # 처음 N개는 상세 로그 출력
            verbose = (processed_count <= verbose_first_n)

            if verbose:
                print(f"\n  [디버깅 {processed_count}] {corp_name} ({ticker})")

            try:
                # 재무제표 수집 (CFS 우선)
                fs_raw = get_dart_fs_quarterly(
                    api_key=api_key,
                    corp_code=corp_code,
                    start_year=start_year,
                    end_year=end_year,
                    fs_div=fs_div,
                    verbose=verbose,
                    session=session
                )

                # CFS 데이터가 없고, OFS 대체 옵션이 켜져 있으면 OFS 시도
                if fs_raw.empty and try_ofs_fallback and fs_div == "CFS":
                    if verbose:
                        print(f"    → CFS 없음, OFS 시도...")

                    fs_raw = get_dart_fs_quarterly(
                        api_key=api_key,
                        corp_code=corp_code,
                        start_year=start_year,
                        end_year=end_year,
                        fs_div="OFS",
                        verbose=verbose,
                        session=session
                    )

                    if not fs_raw.empty:
                        cfs_to_ofs_count += 1
                        if verbose:
                            print(f"    → OFS 데이터 발견!")

                # 데이터가 없으면 스킵
                if fs_raw.empty:
                    no_data_count += 1
                    if verbose:
                        print(f"    → 최종 결과: 데이터 없음")
                    continue

                # 필수 컬럼 추가
                fs_raw["corp_code"] = corp_code
                fs_raw["corp_name"] = corp_name
                fs_raw["ticker"] = ticker  # 참고용으로 넣어도 무방

                # 재무지표 계산 (compute_quarterly_indicators는 사용자 정의 함수로 가정)
                df_ind = compute_quarterly_indicators(fs_raw)

                if not df_ind.empty:
                    # ✅ DB에 저장할 ticker를 FDR Code로 강제 세팅
                    df_ind["ticker"] = ticker

                    batch_indicators.append(df_ind)
                    success_count += 1

                    if verbose or success_count <= 3:
                        print(f"  ✅ [{ticker}] {corp_name}: {len(df_ind)}개 지표 수집")
                else:
                    no_data_count += 1
                    if verbose:
                        print(f"    → 지표 계산 결과 없음")

                # 메모리 정리
                del fs_raw, df_ind

            except Exception as e:
                # 모든 예외를 단순하게 처리
                fail_count += 1
                error_msg = str(e)[:200]
                failed_companies.append({
                    "ticker": ticker,
                    "corp_name": corp_name,
                    "error": error_msg
                })

                # 간단한 오류 메시지만 출력
                print(f"  ❌ [{ticker}] {corp_name}: 오류 발생")

                if verbose:
                    print(f"     → {error_msg}")

                # 연속 오류 시 더 긴 대기
                if fail_count % 5 == 0:
                    wait_time = 20
                    print(f"  ⚠️  연속 오류 {fail_count}회 - {wait_time}초 대기...")
                    time.sleep(wait_time)

                continue

        # 배치 업로드
        if batch_indicators:
            print(f"\n{'='*70}")
            print(f"[BATCH UPLOAD] 배치 {current_batch_num} DB 업로드 중...")

            try:
                # 배치 내 모든 데이터 합치기
                batch_combined = pd.concat(batch_indicators, ignore_index=True)

                # DB 업로드 (upload_indicators_to_db는 사용자 정의 함수로 가정)
                upload_indicators_to_db(batch_combined, db_info, table_name)

                print(f"✅ 배치 업로드 완료: {len(batch_combined)} rows")

                # 메모리 정리
                del batch_combined

            except Exception as e:
                print(f"❌ 배치 업로드 실패: {e}")
                traceback.print_exc()
        else:
            print(f"\n⚠️  배치 {current_batch_num}: 업로드할 데이터 없음")

        # 배치 처리 후 메모리 정리
        del batch_indicators
        gc.collect()

        print(f"{'='*70}")
        print(f"[진행상황] 성공: {success_count} | 데이터없음: {no_data_count} | 실패: {fail_count}")
        if cfs_to_ofs_count > 0:
            print(f"[OFS 대체] {cfs_to_ofs_count}개 기업")
        print(f"{'='*70}")

        # 다음 배치 전 휴식 (마지막 배치는 제외)
        if batch_end < total_companies:
            print(f"\n💤 다음 배치 전 {batch_rest_time}초 휴식...")
            time.sleep(batch_rest_time)
            print()

    # 세션 종료
    session.close()

    # 4) 최종 결과 출력
    print("\n" + "=" * 70)
    print("[최종 결과]")
    print("=" * 70)
    print(f"✅ 성공: {success_count}개 기업")
    print(f"⚠️  데이터 없음: {no_data_count}개 기업")
    print(f"❌ 실패: {fail_count}개 기업")
    if cfs_to_ofs_count > 0:
        print(f"🔄 CFS→OFS 대체: {cfs_to_ofs_count}개 기업")
    print(f"📊 총 처리: {total_companies}개 기업")
    if success_count > 0:
        print(f"📈 성공률: {success_count/total_companies*100:.2f}%")
        if success_count + no_data_count > 0:
            print(f"📉 데이터 확보율: {success_count/(success_count+no_data_count)*100:.2f}%")

    # 5) 실패한 기업 목록 출력
    if failed_companies:
        print("\n" + "=" * 70)
        print("[실패한 기업 목록]")
        print("=" * 70)
        for item in failed_companies[:20]:
            print(f"  • [{item['ticker']}] {item['corp_name']}")
            print(f"    오류: {item['error'][:100]}...")

        if len(failed_companies) > 20:
            print(f"  ... 외 {len(failed_companies)-20}개 기업")

        # 실패 목록을 CSV로 저장
        failed_df = pd.DataFrame(failed_companies)
        failed_csv = f"failed_companies_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")
        print(f"\n💾 실패 목록 저장: {failed_csv}")

    print("=" * 70)
    print("🎉 모든 작업 완료!")
    print("=" * 70)


In [20]:
collect_and_upload_all_companies(
    api_key=API_KEY,
    db_info=db_info,
    start_year=2015,
    end_year=2025,
    fs_div="CFS",
    batch_size=15,
    table_name="korea_fs_data_from_DART",
    listed_only=True,
    verbose_first_n=3,
    try_ofs_fallback=True,
    batch_rest_time=30,
    use_fdr_filter=True
)


[INFO] HTTP 세션 생성 완료 (재시도 로직 + User-Agent 설정)
[STEP 1] DART 기업 목록 로드 중...
[INFO] DART 기준 상장사만 1차 필터링 완료
      → 남은 기업 수: 3906

[STEP 1-2] FinanceDataReader에서 현재 상장 종목 코드 로드 중 (KRX)...
[WARNING] FDR 데이터에 'Type' 컬럼이 없어 Name 기반 필터를 사용합니다.
[INFO] Name 기반 필터링 (ETF/ETN/리츠/스팩 추정 제거)
      → 필터 전: 2881개, 필터 후: 2780개
[INFO] FDR KRX 현재 상장 일반 주식 수(필터 후): 2780
[INFO] FDR 기준 현재 상장 일반 주식으로 2차 필터링 완료
      → 필터 전: 3906개, 필터 후: 2662개

[INFO] 최종 대상 기업 수: 2662개 (현재 상장 일반 주식 기준)

[상장사 샘플]
      corp_name stock_code
36636       GRT     900290
41823       로스웰     900260
48350    맥쿼리인프라     088980
48630   크리스탈신소재     900250
51444        우진     105840
51563      인화정공     101930
51576      대원산업     005710
51577        대동     000490
51805   삼화콘덴서공업     001820
51855       유니온     000910

[STEP 2] 재무데이터 수집 시작
[설정] 배치크기=15, 배치간휴식=30초
[설정] fs_div=CFS, OFS대체=True
[설정] API대기시간=1.0~2.0초(랜덤)

[BATCH 1/178] 처리 중: 1~15/2662


기업 처리:   0%|          | 0/15 [00:00<?, ?it/s]


  [디버깅 1] GRT (900290)
    [API] 2015-Q1: status=013, msg=조회된 데이타가 없습니다.
    [API] 2015-Q2: status=013, msg=조회된 데이타가 없습니다.
    [API] 2015-Q3: status=013, msg=조회된 데이타가 없습니다.
    [API] 2015-Q4: status=013, msg=조회된 데이타가 없습니다.
    [API] 2016-Q1: status=013, msg=조회된 데이타가 없습니다.
    [API] 2016-Q3: status=013, msg=조회된 데이타가 없습니다.
    [API] 2016-Q4: status=013, msg=조회된 데이타가 없습니다.
    [API] 2017-Q1: status=013, msg=조회된 데이타가 없습니다.
    [API] 2018-Q3: status=013, msg=조회된 데이타가 없습니다.
    [API] 2025-Q1: status=013, msg=조회된 데이타가 없습니다.
    [API] 2025-Q2: status=013, msg=조회된 데이타가 없습니다.


기업 처리:   7%|▋         | 1/15 [00:51<12:03, 51.66s/it]

  ✅ [900290] GRT: 891개 지표 수집

  [디버깅 2] 로스웰 (900260)
    [API] 2015-Q1: status=013, msg=조회된 데이타가 없습니다.
    [API] 2015-Q2: status=013, msg=조회된 데이타가 없습니다.
    [API] 2015-Q3: status=013, msg=조회된 데이타가 없습니다.
    [API] 2015-Q4: status=013, msg=조회된 데이타가 없습니다.
    [API] 2016-Q1: status=013, msg=조회된 데이타가 없습니다.
    [API] 2025-Q3: status=013, msg=조회된 데이타가 없습니다.
    [API] 2025-Q4: status=013, msg=조회된 데이타가 없습니다.


...


[BATCH UPLOAD] 배치 10 DB 업로드 중...
업로드 완료: 16728 rows
✅ 배치 업로드 완료: 16728 rows
[진행상황] 성공: 138 | 데이터없음: 12 | 실패: 0
[OFS 대체] 11개 기업

💤 다음 배치 전 30초 휴식...


[BATCH 11/178] 처리 중: 151~165/2662


기업 처리: 100%|██████████| 15/15 [14:50<00:00, 59.34s/it]



[BATCH UPLOAD] 배치 11 DB 업로드 중...
업로드 완료: 16378 rows
✅ 배치 업로드 완료: 16378 rows
[진행상황] 성공: 152 | 데이터없음: 13 | 실패: 0
[OFS 대체] 14개 기업

💤 다음 배치 전 30초 휴식...


[BATCH 12/178] 처리 중: 166~180/2662


기업 처리:  87%|████████▋ | 13/15 [37:21<05:44, 172.40s/it]


KeyboardInterrupt: 

In [21]:
def truncate_table(db_info, table_name="korea_fs_data_from_DART"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", 3306),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=True,  # TRUNCATE는 바로 커밋
    )

    try:
        with conn.cursor() as cur:
            sql = f"TRUNCATE TABLE {table_name};"
            cur.execute(sql)
        print(f"✅ 테이블 초기화 완료: {table_name}")
    finally:
        conn.close()


# 실행
truncate_table(db_info, "korea_fs_data_from_DART")

✅ 테이블 초기화 완료: korea_fs_data_from_DART


In [37]:
# 1) 배당 관련 계정이 실제로 뭐라고 나오는지 확인
div_rows = fs_raw[fs_raw["account_nm"].astype(str).str.contains("배당", na=False)]

print("배당 관련 계정 행 수:", len(div_rows))
print(div_rows[["bsns_year", "reprt_code", "sj_div", "account_id", "account_nm", "thstrm_amount"]]
      .drop_duplicates()
      .sort_values(["bsns_year", "reprt_code"])
      .head(50)
)

배당 관련 계정 행 수: 261
     bsns_year reprt_code sj_div  \
88        2015      11011     CF   
112       2015      11011     CF   
152       2015      11011    SCE   
153       2015      11011    SCE   
154       2015      11011    SCE   
835       2016      11011     CF   
859       2016      11011     CF   
899       2016      11011    SCE   
900       2016      11011    SCE   
902       2016      11011    SCE   
460       2016      11012     CF   
484       2016      11012     CF   
524       2016      11012    SCE   
525       2016      11012    SCE   
527       2016      11012    SCE   
273       2016      11013     CF   
297       2016      11013     CF   
337       2016      11013    SCE   
338       2016      11013    SCE   
340       2016      11013    SCE   
647       2016      11014     CF   
671       2016      11014     CF   
711       2016      11014    SCE   
712       2016      11014    SCE   
714       2016      11014    SCE   
1579      2017      11011     CF   
1605      